<a href="https://colab.research.google.com/github/Mounyasri989/Carbon_Coach_AI_Agent/blob/main/Carbon_Coach_AI_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [31]:
!pip install -q google-adk gradio pandas matplotlib

In [32]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
import gradio as gr

from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

from google.adk.tools import ToolContext

print("Libraries installed successfully!")

Libraries installed successfully!


In [33]:
from getpass import getpass

GOOGLE_API_KEY = getpass("Enter your Gemini API Key: ")

os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "FALSE"

print("Gemini API configured!")

Enter your Gemini API Key: ··········
Gemini API configured!


In [34]:
EMISSION_FACTORS = {

    "travel": {
        "bus": 0.10,
        "car": 0.17,
        "bike": 0.10,
        "train": 0.04,
        "walking": 0.0,
        "cycling": 0.0
    },

    "electricity": {
        "grid": 0.710
    },

    "food": {
        "vegetarian": 0.5,
        "non_vegetarian": 2.5,
        "vegan": 0.4
    },

    "waste": {
        "general": 0.5
    }
}

FACTOR_SOURCE = {
    "travel": "Illustrative demonstration factors; replace with "
              "official mode-specific factors before final evaluation.",

    "electricity": "Illustrative Indian grid factor. Consult the "
                   "CEA CO2 Baseline Database for the applicable year.",

    "food": "Illustrative meal-level assumptions, not measured values.",

    "waste": "Illustrative waste-management assumption; depends "
             "on waste type and disposal method."
}

print("Emission factors loaded!")

Emission factors loaded!


In [35]:
def calculate_travel_emissions(mode: str, km: float):

    mode = mode.lower()

    if mode not in EMISSION_FACTORS["travel"]:
        return {"error": "Unsupported travel mode"}

    if km < 0:
        return {"error": "Distance cannot be negative"}

    factor = EMISSION_FACTORS["travel"][mode]

    emissions = km * factor

    return {
        "category": "travel",
        "mode": mode,
        "distance_km": km,
        "emission_factor": factor,
        "emissions_kg": round(emissions, 3),
        "unit": "kg CO2e",
        "source": FACTOR_SOURCE["travel"]
    }

In [36]:
def calculate_electricity_emissions(kwh: float):

    if kwh < 0:
        return {"error": "Electricity cannot be negative"}

    factor = EMISSION_FACTORS["electricity"]["grid"]

    emissions = kwh * factor

    return {
        "category": "electricity",
        "consumption_kwh": kwh,
        "emission_factor": factor,
        "emissions_kg": round(emissions, 3),
        "unit": "kg CO2e",
        "source": FACTOR_SOURCE["electricity"]
    }

In [37]:
def calculate_food_emissions(meal_type: str):

    meal_type = meal_type.lower()

    if meal_type not in EMISSION_FACTORS["food"]:
        return {"error": "Unsupported meal type"}

    factor = EMISSION_FACTORS["food"][meal_type]

    return {
        "category": "food",
        "meal_type": meal_type,
        "emissions_kg": factor,
        "unit": "kg CO2e",
        "source": FACTOR_SOURCE["food"]
    }

In [38]:
def calculate_waste_emissions(waste_kg: float):

    if waste_kg < 0:
        return {"error": "Waste quantity cannot be negative"}

    factor = EMISSION_FACTORS["waste"]["general"]

    emissions = waste_kg * factor

    return {
        "category": "waste",
        "waste_kg": waste_kg,
        "emissions_kg": round(emissions, 3),
        "unit": "kg CO2e",
        "source": FACTOR_SOURCE["waste"]
    }

In [39]:
def log_activity(
    category: str,
    details: str,
    emissions_kg: float,
    tool_context: ToolContext
):
    """
    Save calculated activity emissions in session memory.
    """

    if emissions_kg < 0:
        return {"error": "Emissions cannot be negative"}

    activities = tool_context.state.get("activities", [])

    activity = {
        "category": category,
        "details": details,
        "emissions_kg": emissions_kg
    }

    activities.append(activity)

    tool_context.state["activities"] = activities

    return {
        "status": "Activity saved",
        "activity": activity,
        "total_activities": len(activities)
    }

In [40]:
def get_summary(period: str, tool_context: ToolContext):

    activities = tool_context.state.get("activities", [])

    total = sum(
        item["emissions_kg"]
        for item in activities
    )

    category_totals = {}

    for item in activities:

        category = item["category"]

        category_totals[category] = (
            category_totals.get(category, 0)
            + item["emissions_kg"]
        )

    return {
        "period_requested": period,
        "number_of_activities": len(activities),
        "total_emissions_kg": round(total, 3),
        "category_breakdown": {
            k: round(v, 3)
            for k, v in category_totals.items()
        },
        "activities": activities
    }

In [41]:
def suggest_reductions(tool_context: ToolContext):

    activities = tool_context.state.get("activities", [])

    totals = {}

    for item in activities:

        category = item["category"]

        totals[category] = (
            totals.get(category, 0)
            + item["emissions_kg"]
        )

    suggestions = []

    if totals.get("travel", 0) > 0:

        suggestions.append({
            "action": "Consider public transport, cycling or carpooling.",
            "reason": "You have recorded travel-related emissions."
        })

    if totals.get("electricity", 0) > 0:

        suggestions.append({
            "action": "Reduce unnecessary electricity usage.",
            "reason": "You have recorded electricity-related emissions."
        })

    if totals.get("food", 0) > 0:

        suggestions.append({
            "action": "Consider lower-impact meal choices.",
            "reason": "You have recorded food-related emissions."
        })

    if totals.get("waste", 0) > 0:

        suggestions.append({
            "action": "Reduce waste and separate recyclable materials.",
            "reason": "You have recorded waste-related emissions."
        })

    return {
        "suggestions": suggestions,
        "note": "Potential savings have not been quantified. "
                "They require a comparison with a defined alternative."
    }

In [42]:
root_agent = Agent(

    name="carbon_coach",

    model="gemini-3.5-flash-lite",

    description="""
    An AI agent that measures, analyzes and helps
    users reduce their carbon footprint.
    """,

    instruction="""

    You are Carbon Coach, an intelligent environmental assistant.

    Your responsibilities:

    1. Understand daily activities described in natural language.
    2. Identify travel, electricity, food and waste activities.
    3. Call the appropriate Python calculation tools.
    4. Never calculate emissions mentally or invent factors.
    5. Log every successfully calculated activity.
    6. Remember activities using session memory.
    7. Answer questions about total emissions and category breakdowns.
    8. Provide personalized reduction recommendations.

    IMPORTANT RULES:

    - Always use calculation tools for numerical emissions.
    - Never invent carbon emission numbers.
    - Always report emissions in kg CO2e.
    - Show the emission factor and its source when relevant.
    - If necessary information is missing, ask the user.
    - Do not assume AC electricity consumption from operating hours.
    - Do not claim estimated savings are actual reductions.
    - Use session memory for activity history.
    - When users ask for summaries, use get_summary.
    - When users ask for suggestions, use suggest_reductions.

    Respond in simple and understandable language.

    """,

    tools=[
        calculate_travel_emissions,
        calculate_electricity_emissions,
        calculate_food_emissions,
        calculate_waste_emissions,
        log_activity,
        get_summary,
        suggest_reductions
    ]
)

print("Carbon Coach ADK Agent created!")

Carbon Coach ADK Agent created!


In [43]:
APP_NAME = "carbon_coach_app"

USER_ID = "student_001"

SESSION_ID = "carbon_session_001"

session_service = InMemorySessionService()

runner = Runner(
    agent=root_agent,
    app_name=APP_NAME,
    session_service=session_service
)

print("ADK Runner initialized!")

ADK Runner initialized!


In [44]:
import asyncio

session = await session_service.create_session(
    app_name=APP_NAME,
    user_id=USER_ID,
    session_id=SESSION_ID
)

print("Carbon Coach session created!")

Carbon Coach session created!


In [45]:
async def ask_carbon_coach(message):

    content = types.Content(
        role="user",
        parts=[
            types.Part(text=message)
        ]
    )

    final_response = ""

    async for event in runner.run_async(
        user_id=USER_ID,
        session_id=SESSION_ID,
        new_message=content
    ):

        if event.is_final_response():

            if event.content and event.content.parts:

                final_response = "".join(
                    part.text
                    for part in event.content.parts
                    if part.text
                )

    return final_response

In [49]:
response = await ask_carbon_coach(
    "I travelled 12 km by bus today. "
    "Calculate emissions using the bus emission factor "
    "and save 12 km as my travel activity."
)
print(response)

Your travel emissions for 12 km by bus are **1.2 kg CO2e**.

* **Emission factor:** 0.1 kg CO2e per km
* **Source:** Illustrative demonstration factors; replace with official mode-specific factors before final evaluation.

This activity has been successfully saved to your session history.


In [50]:
# TEST 1: Individual emission calculators

print("TESTING CARBON CALCULATORS")

# 1. Travel: 12 km by bus
travel_result = calculate_travel_emissions("bus", 12)
print("\n1. Travel emissions (12 km bus):")
print(travel_result)

# 2. Electricity: 10 kWh
electricity_result = calculate_electricity_emissions(10)
print("\n2. Electricity emissions (10 kWh):")
print(electricity_result)

# 3. Food: one non-vegetarian meal
food_result = calculate_food_emissions("nonveg")
print("\n3. Food emissions (1 nonveg meal):")
print(food_result)

# 4. Waste: 2 kg
waste_result = calculate_waste_emissions(2)
print("\n4. Waste emissions (2 kg):")
print(waste_result)

TESTING CARBON CALCULATORS

1. Travel emissions (12 km bus):
{'category': 'travel', 'mode': 'bus', 'distance_km': 12, 'emission_factor': 0.1, 'emissions_kg': 1.2, 'unit': 'kg CO2e', 'source': 'Illustrative demonstration factors; replace with official mode-specific factors before final evaluation.'}

2. Electricity emissions (10 kWh):
{'category': 'electricity', 'consumption_kwh': 10, 'emission_factor': 0.71, 'emissions_kg': 7.1, 'unit': 'kg CO2e', 'source': 'Illustrative Indian grid factor. Consult the CEA CO2 Baseline Database for the applicable year.'}

3. Food emissions (1 nonveg meal):
{'error': 'Unsupported meal type'}

4. Waste emissions (2 kg):
{'category': 'waste', 'waste_kg': 2, 'emissions_kg': 1.0, 'unit': 'kg CO2e', 'source': 'Illustrative waste-management assumption; depends on waste type and disposal method.'}


In [51]:
# TEST 2: Save four activities through the AI agent

test_messages = [
    "I travelled 12 km by bus today. Calculate my emissions and save this activity.",
    "I used 10 kWh of electricity today. Calculate my emissions and save this activity.",
    "I ate one non-vegetarian meal today. Calculate my emissions and save this activity.",
    "I generated 2 kg of waste today. Calculate my emissions and save this activity."
]

for message in test_messages:
    print("\nUSER:", message)

    response = await ask_carbon_coach(message)

    print("CARBON COACH:", response)


USER: I travelled 12 km by bus today. Calculate my emissions and save this activity.
CARBON COACH: Your travel emissions for 12 km by bus are **1.2 kg CO2e**.

* **Emission factor:** 0.1 kg CO2e per km
* **Source:** Illustrative demonstration factors; replace with official mode-specific factors before final evaluation.

This activity has been successfully saved to your session history.

USER: I used 10 kWh of electricity today. Calculate my emissions and save this activity.
CARBON COACH: Your electricity emissions for 10 kWh are **7.1 kg CO2e**.

* **Emission factor:** 0.71 kg CO2e per kWh
* **Source:** Illustrative Indian grid factor. Consult the CEA CO2 Baseline Database for the applicable year.

This activity has been successfully saved to your session history.

USER: I ate one non-vegetarian meal today. Calculate my emissions and save this activity.
CARBON COACH: I'm sorry, but "non-vegetarian" is not currently a supported meal type for calculation. Supported meal types typically 

In [52]:
# TEST 3: Check saved activities and total emissions

response = await ask_carbon_coach(
    "Show me all the activities I have saved in this session. "
    "Give me the emissions for each activity and the grand total "
    "in kg CO2e."
)

print(response)

Here is the summary of all the activities you have saved in this session:

### Saved Activities:
1. **Travel:** 12 km by bus — **1.2 kg CO2e**
2. **Travel:** 15 km by bus — **1.5 kg CO2e**
3. **Travel:** 0 km by bus — **0.0 kg CO2e**
4. **Travel:** 12 km by bus — **1.2 kg CO2e**
5. **Travel:** 12 km by bus — **1.2 kg CO2e**
6. **Electricity:** 10 kWh electricity usage — **7.1 kg CO2e**
7. **Waste:** 2 kg of waste generated — **1.0 kg CO2e**

---

### Category Breakdown:
* **Electricity:** 7.1 kg CO2e
* **Travel:** 5.1 kg CO2e
* **Waste:** 1.0 kg CO2e

### **Grand Total Emissions:** **13.2 kg CO2e**


In [53]:
# TEST 4: Ask for personalized carbon reduction tips

response = await ask_carbon_coach(
    "Analyze my saved activities. "
    "Suggest 5 practical ways to reduce my carbon footprint. "
    "Prioritize the activities causing the most emissions. "
    "Estimate the kg CO2e savings for each suggestion "
    "using the emission factors available to you."
)

print(response)

Based on your recorded session activities (totaling **13.2 kg CO2e**), here is an analysis and practical reduction suggestions prioritized by your highest-emitting categories:

---

### 1. Electricity Usage (Highest Impact: 7.1 kg CO2e)
* **Action:** Reduce electricity consumption (e.g., turning off lights/appliances when not in use, shifting to energy-efficient LED bulbs, or adjusting air conditioning set points).
* **Why:** Electricity is your largest emission source, accounting for over **53%** of your total footprint (7.1 kg CO2e out of 13.2 kg CO2e).
* **Estimated Savings:** *Note: Potential savings are dependent on how many kWh you reduce compared to your current usage baseline. For example, reducing your 10 kWh consumption by half (5 kWh) would save approximately **3.55 kg CO2e**.*

### 2. Bus Travel (Second Highest Impact: 5.1 kg CO2e total across 5 trips)
* **Action:** Combine trips or replace motorized bus travel with active transport (walking or cycling) for shorter distance

In [55]:
async def chat_interface(message, history):

    if not message.strip():
        return "Please enter your activity."

    response = await ask_carbon_coach(message)

    return response


with gr.Blocks(
    theme=gr.themes.Soft(),
    title="Carbon Coach"
) as demo:

    gr.Markdown(
        """
        # 🌱 Carbon Coach

        ### Your Personal AI Carbon Footprint Assistant

        Measure. Understand. Reduce.
        """
    )

    gr.Markdown(
        """
        Describe your daily activities in natural language.

        Example:
        - I travelled 12 km by bus.
        - I consumed 5 kWh electricity.
        - I ate a vegetarian meal.
        - Show my total carbon footprint.
        - Suggest ways to reduce emissions.
        """
    )

    gr.ChatInterface(
        fn=chat_interface,
        title="Talk to Carbon Coach"
    )


demo.launch(share=True)

/tmp/ipykernel_2047/2540460705.py:11: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://82ddb4bae4e986b636.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [56]:
golden_tests = [

    {
        "name": "Bus travel",
        "actual": calculate_travel_emissions("bus", 10),
        "expected": 1.0
    },

    {
        "name": "Car travel",
        "actual": calculate_travel_emissions("car", 10),
        "expected": 1.7
    },

    {
        "name": "Electricity",
        "actual": calculate_electricity_emissions(10),
        "expected": 7.1
    },

    {
        "name": "Walking",
        "actual": calculate_travel_emissions("walking", 5),
        "expected": 0.0
    }
]


for test in golden_tests:

    actual = test["actual"]["emissions_kg"]

    expected = test["expected"]

    passed = abs(actual - expected) < 0.001

    print(
        test["name"],
        "| Expected:", expected,
        "| Actual:", actual,
        "| Result:", "PASS" if passed else "FAIL"
    )

Bus travel | Expected: 1.0 | Actual: 1.0 | Result: PASS
Car travel | Expected: 1.7 | Actual: 1.7 | Result: PASS
Electricity | Expected: 7.1 | Actual: 7.1 | Result: PASS
Walking | Expected: 0.0 | Actual: 0.0 | Result: PASS
